# 14. Politica Final de Despliegue Thoracolumbar - Colab

Este notebook cierra la etapa experimental y convierte los hallazgos del
proyecto en una **decision final operativa** para el pipeline de identificacion
de vertebras toracicas y lumbares.

## Contexto resumido

- `07` produjo el mejor pipeline validado con metrica final de segmentacion
- `12` mejoro metricas intermedias del estimador `last_visible`, pero degrado la
  mascara final
- `13` mostro que el refinado parece util en algunos subcasos, pero no de forma
  lo bastante estable como para reemplazar al baseline completo

## Objetivo del notebook

1. comparar politicas finales de despliegue
2. imponer un criterio conservador alineado con el objetivo real del proyecto
3. declarar la politica final recomendada
4. exportar una tabla clara para el documento final

## Principio rector

En esta etapa ya no buscamos optimizar una metrica intermedia a cualquier costo.
Priorizamos:

1. no aumentar el riesgo de perder vertebras correctas
2. mantener una politica interpretable y robusta
3. solo aceptar cambios si aportan mejora real y defendible

## 0. Preparacion de Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/DataRadriografias")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"No existe la carpeta: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

## 1. Librerias y rutas

Este notebook reutiliza principalmente los resultados del notebook `13`, que ya
consolido la comparacion baseline/refined y varias politicas hibridas.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'analysis_outputs' / 'final_deployment_policy_thoracolumbar'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def resolve_existing(*relative_candidates: str) -> Path:
    search_roots = [ROOT, ROOT / 'reports']
    for base in search_roots:
        for rel in relative_candidates:
            candidate = base / rel
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'No se encontro ninguno de estos archivos: {relative_candidates}')


POLICY_COMPARE_PATH = resolve_existing(
    'analysis_outputs/hybrid_conservative_clipping_policy_thoracolumbar/hybrid_policy_compare.csv'
)
POLICY_MERGED_PATH = resolve_existing(
    'analysis_outputs/hybrid_conservative_clipping_policy_thoracolumbar/hybrid_policy_merged_compare.csv'
)
POLICY_REFINED_USEFUL_PATH = resolve_existing(
    'analysis_outputs/hybrid_conservative_clipping_policy_thoracolumbar/hybrid_policy_refined_useful_cases.csv'
)
POLICY_RECOMMENDATION_PATH = resolve_existing(
    'analysis_outputs/hybrid_conservative_clipping_policy_thoracolumbar/hybrid_policy_recommendation.csv'
)
BASELINE_REFINED_CONTEXT_PATH = resolve_existing(
    'analysis_outputs/hybrid_conservative_clipping_policy_thoracolumbar/baseline_vs_refined_context.csv'
)

print('OUTPUT_DIR:', OUTPUT_DIR)

## 2. Carga de resultados previos

In [ ]:
policy_compare_df = pd.read_csv(POLICY_COMPARE_PATH)
policy_merged_df = pd.read_csv(POLICY_MERGED_PATH)
refined_useful_df = pd.read_csv(POLICY_REFINED_USEFUL_PATH)
prior_recommendation_df = pd.read_csv(POLICY_RECOMMENDATION_PATH)
context_df = pd.read_csv(BASELINE_REFINED_CONTEXT_PATH)

if 'split' not in refined_useful_df.columns:
    if 'manifest_split' in refined_useful_df.columns:
        refined_useful_df['split'] = refined_useful_df['manifest_split']
    else:
        refined_useful_df['split'] = 'unknown'

display(context_df)
display(policy_compare_df)
display(prior_recommendation_df)

## 3. Criterio conservador de seleccion final

A diferencia del notebook `13`, aqui ya no basta con maximizar un proxy.

Definimos un criterio final mas estricto:

- una politica no debe empeorar `mean_missing_count` frente al baseline
- si una politica mejora solo marginalmente el proxy pero aumenta mucho el riesgo,
  no se adopta
- se prefiere una politica simple antes que una compleja, si el beneficio no es
  claramente superior

In [ ]:
baseline_row = policy_compare_df.loc[policy_compare_df['policy_name'] == 'baseline_always'].iloc[0]
raw_row = policy_compare_df.loc[policy_compare_df['policy_name'] == 'raw_only'].iloc[0]

policy_eval_df = policy_compare_df.copy()
policy_eval_df['baseline_proxy'] = float(baseline_row['mean_proxy_dice'])
policy_eval_df['baseline_missing'] = float(baseline_row['mean_missing_count'])
policy_eval_df['baseline_extra'] = float(baseline_row['mean_extra_count'])
policy_eval_df['proxy_gain_vs_baseline'] = (
    policy_eval_df['mean_proxy_dice'] - policy_eval_df['baseline_proxy']
)
policy_eval_df['missing_change_vs_baseline'] = (
    policy_eval_df['mean_missing_count'] - policy_eval_df['baseline_missing']
)
policy_eval_df['extra_change_vs_baseline'] = (
    policy_eval_df['mean_extra_count'] - policy_eval_df['baseline_extra']
)
policy_eval_df['is_conservative_candidate'] = (
    (policy_eval_df['mean_missing_count'] <= policy_eval_df['baseline_missing'])
    & (policy_eval_df['fraction_refined'] <= 0.50)
)
policy_eval_df['net_conservative_score'] = (
    policy_eval_df['proxy_gain_vs_baseline']
    - (policy_eval_df['missing_change_vs_baseline'].clip(lower=0) * 0.75)
    - (policy_eval_df['extra_change_vs_baseline'].clip(lower=0) * 0.10)
    + (policy_eval_df['clean_improvement_rate'] * 0.05)
    - (policy_eval_df['harm_rate'] * 0.05)
)

display(
    policy_eval_df[
        [
            'policy_name', 'mean_proxy_dice', 'mean_missing_count', 'mean_extra_count',
            'fraction_raw', 'fraction_baseline', 'fraction_refined',
            'proxy_gain_vs_baseline', 'missing_change_vs_baseline',
            'extra_change_vs_baseline', 'is_conservative_candidate',
            'net_conservative_score'
        ]
    ].sort_values(
        ['is_conservative_candidate', 'net_conservative_score', 'mean_proxy_dice'],
        ascending=[False, False, False],
    )
)

## 4. Politicas finales candidatas

Aqui hacemos una lectura mas humana de las opciones:

- `baseline_always`
- la mejor politica proxy absoluta
- la mejor politica conservadora
- `raw_only` como control

In [ ]:
best_proxy_row = policy_eval_df.sort_values(
    ['mean_proxy_dice', 'mean_missing_count', 'mean_extra_count'],
    ascending=[False, True, True],
).iloc[0]

conservative_candidates_df = policy_eval_df.loc[policy_eval_df['is_conservative_candidate']].copy()
if conservative_candidates_df.empty:
    best_conservative_row = baseline_row.copy()
else:
    best_conservative_row = conservative_candidates_df.sort_values(
        ['net_conservative_score', 'mean_proxy_dice', 'mean_missing_count'],
        ascending=[False, False, True],
    ).iloc[0]

final_candidates_df = pd.DataFrame([
    {
        'candidate_role': 'best_proxy_policy',
        'policy_name': best_proxy_row['policy_name'],
        'mean_proxy_dice': float(best_proxy_row['mean_proxy_dice']),
        'mean_missing_count': float(best_proxy_row['mean_missing_count']),
        'mean_extra_count': float(best_proxy_row['mean_extra_count']),
        'fraction_refined': float(best_proxy_row['fraction_refined']),
    },
    {
        'candidate_role': 'best_conservative_policy',
        'policy_name': best_conservative_row['policy_name'],
        'mean_proxy_dice': float(best_conservative_row['mean_proxy_dice']),
        'mean_missing_count': float(best_conservative_row['mean_missing_count']),
        'mean_extra_count': float(best_conservative_row['mean_extra_count']),
        'fraction_refined': float(best_conservative_row['fraction_refined']),
    },
    {
        'candidate_role': 'baseline_reference',
        'policy_name': baseline_row['policy_name'],
        'mean_proxy_dice': float(baseline_row['mean_proxy_dice']),
        'mean_missing_count': float(baseline_row['mean_missing_count']),
        'mean_extra_count': float(baseline_row['mean_extra_count']),
        'fraction_refined': float(baseline_row['fraction_refined']),
    },
    {
        'candidate_role': 'raw_reference',
        'policy_name': raw_row['policy_name'],
        'mean_proxy_dice': float(raw_row['mean_proxy_dice']),
        'mean_missing_count': float(raw_row['mean_missing_count']),
        'mean_extra_count': float(raw_row['mean_extra_count']),
        'fraction_refined': float(raw_row['fraction_refined']),
    },
])

display(final_candidates_df)

## 5. Decision final del proyecto

Regla de decision:

- si la mejor politica conservadora supera claramente al baseline sin aumentar el
  riesgo, se adopta
- si la evidencia no es suficientemente fuerte, se mantiene el baseline como
  politica final

Como el notebook `13` usa un **proxy de labels** y no una medicion completa de
segmentacion pixel a pixel, imponemos un margen minimo de seguridad antes de
reemplazar el baseline. Esto evita sobreinterpretar mejoras pequenas.

In [ ]:
MIN_PROXY_GAIN_TO_REPLACE_BASELINE = 0.005

adopt_conservative = (
    (best_conservative_row['policy_name'] != 'baseline_always')
    and (best_conservative_row['mean_missing_count'] <= baseline_row['mean_missing_count'])
    and (best_conservative_row['mean_extra_count'] <= baseline_row['mean_extra_count'])
    and (best_conservative_row['harm_rate'] <= baseline_row['harm_rate'])
    and (
        best_conservative_row['mean_proxy_dice']
        >= baseline_row['mean_proxy_dice'] + MIN_PROXY_GAIN_TO_REPLACE_BASELINE
    )
)

if adopt_conservative:
    final_policy_name = str(best_conservative_row['policy_name'])
    final_reason = (
        'La mejor politica conservadora supera al baseline con margen suficiente '
        'sin aumentar el riesgo principal de missing labels.'
    )
else:
    final_policy_name = 'baseline_always'
    final_reason = (
        'La evidencia no demuestra una mejora lo bastante robusta y segura como '
        'para reemplazar el baseline validado con segmentacion final.'
    )

final_decision_df = pd.DataFrame([
    {
        'decision_type': 'final_operational_policy',
        'selected_policy': final_policy_name,
        'reason': final_reason,
        'reference_value': float(MIN_PROXY_GAIN_TO_REPLACE_BASELINE),
    },
    {
        'decision_type': 'best_proxy_policy',
        'selected_policy': str(best_proxy_row['policy_name']),
        'reason': 'Sirve como evidencia exploratoria, pero no necesariamente como politica final.',
        'reference_value': float(best_proxy_row['mean_proxy_dice']),
    },
    {
        'decision_type': 'best_conservative_policy',
        'selected_policy': str(best_conservative_row['policy_name']),
        'reason': 'Es la mejor politica bajo restricciones de seguridad y simplicidad.',
        'reference_value': float(best_conservative_row['mean_proxy_dice']),
    },
])

display(final_decision_df)

## 6. Implicaciones para el modelo final

Esta seccion deja por escrito que modelo y que politica deben considerarse como
salida final del proyecto.

In [ ]:
final_model_summary_df = pd.DataFrame([
    {
        'component': 'binary_spine_localizer',
        'selected_variant': 'binary_spine_thoracolumbar_best.pt',
        'status': 'final',
        'justification': 'Localiza la ROI y ya estaba estable en el pipeline.',
    },
    {
        'component': 'multiclass_thoracolumbar_segmenter',
        'selected_variant': 'thoracolumbar_partial_cascade_explained_best.pt',
        'status': 'final',
        'justification': 'Es la mejor base multiclase usada por las etapas posteriores.',
    },
    {
        'component': 'last_visible_estimator',
        'selected_variant': 'last_visible_estimator_thoracolumbar_best.pt',
        'status': 'final',
        'justification': 'El refinado mejoro metricas intermedias, pero no la metrica final del pipeline.',
    },
    {
        'component': 'clipping_policy',
        'selected_variant': final_policy_name,
        'status': 'final',
        'justification': final_reason,
    },
])

display(final_model_summary_df)

## 7. Razonamiento final del proyecto

Esta tabla resume por que la recomendacion final se mantiene conservadora.
Nos interesa dejar trazabilidad de las decisiones para el informe y para una
futura iteracion del proyecto.

In [ ]:
final_rationale_df = pd.DataFrame([
    {
        'dimension': 'objetivo_principal',
        'decision': 'priorizar identificacion estable de vertebras visibles',
        'argument': 'La metrica final del proyecto depende de no eliminar vertebras correctas durante el clipping.',
    },
    {
        'dimension': 'resultado_de_12',
        'decision': 'no adoptar refined retraining como reemplazo global',
        'argument': 'Aunque mejoro el estimador last_visible, empeoro la salida final de segmentacion en la comparacion validada.',
    },
    {
        'dimension': 'resultado_de_13',
        'decision': 'usar politica hibrida solo como evidencia exploratoria',
        'argument': 'Los beneficios del refined aparecen en algunos subgrupos, pero no con estabilidad suficiente para reemplazar el baseline en despliegue general.',
    },
    {
        'dimension': 'politica_final',
        'decision': final_policy_name,
        'argument': final_reason,
    },
    {
        'dimension': 'criterio_de_despliegue',
        'decision': 'preferir simpleza e interpretabilidad',
        'argument': 'Una regla simple y validada es mas defendible que una politica mas compleja sin ganancia final consistente.',
    },
])

display(final_rationale_df)

## 8. Hallazgos relevantes para el documento final

In [ ]:
refined_useful_summary_df = pd.DataFrame([
    {'metric': 'refined_useful_cases_total', 'value': int(len(refined_useful_df))},
    {
        'metric': 'refined_useful_cases_scoliosis_ratio',
        'value': float((refined_useful_df['split'].astype(str).str.lower() == 'scoliosis').mean()) if len(refined_useful_df) > 0 else np.nan,
    },
    {
        'metric': 'refined_useful_cases_mean_gap',
        'value': float(pd.to_numeric(refined_useful_df['baseline_refined_idx_gap'], errors='coerce').mean()) if len(refined_useful_df) > 0 else np.nan,
    },
])

display(refined_useful_summary_df)
display(refined_useful_df.head(20))

## 9. Checklist de cierre tecnico

Esta seccion deja claro que elementos deben citarse como definitivos al
continuar con inferencia, demostracion o documento final.

In [ ]:
deployment_checklist_df = pd.DataFrame([
    {
        'step': 1,
        'item': 'modelo binario de columna',
        'status': 'usar version validada',
        'detail': 'binary_spine_thoracolumbar_best.pt para localizar la ROI inicial.',
    },
    {
        'step': 2,
        'item': 'modelo multiclase thoracolumbar',
        'status': 'usar version validada',
        'detail': 'thoracolumbar_partial_cascade_explained_best.pt como segmentador base.',
    },
    {
        'step': 3,
        'item': 'estimador last visible',
        'status': 'usar baseline validado',
        'detail': 'last_visible_estimator_thoracolumbar_best.pt y no el refined retraining como reemplazo global.',
    },
    {
        'step': 4,
        'item': 'politica de clipping',
        'status': 'seleccion final',
        'detail': f'Aplicar {final_policy_name} como politica operativa del pipeline.',
    },
    {
        'step': 5,
        'item': 'uso de notebooks 12 y 13',
        'status': 'evidencia experimental',
        'detail': 'Mantenerlos como soporte de decisiones, no como sustitutos automaticos del pipeline final.',
    },
])

display(deployment_checklist_df)

## 10. Visualizaciones finales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(policy_eval_df['policy_name'], policy_eval_df['mean_proxy_dice'], color='#2f6db3')
axes[0].set_title('Proxy por politica candidata')
axes[0].tick_params(axis='x', rotation=80)
axes[0].grid(alpha=0.25, axis='y')

axes[1].bar(policy_eval_df['policy_name'], policy_eval_df['mean_missing_count'], color='#c46b2d')
axes[1].set_title('Missing labels por politica candidata')
axes[1].tick_params(axis='x', rotation=80)
axes[1].grid(alpha=0.25, axis='y')

plt.tight_layout()
plt.show()

## 11. Exportacion final

Estas tablas estan pensadas para alimentar directamente el documento final del
proyecto.

In [ ]:
policy_eval_path = OUTPUT_DIR / 'final_policy_evaluation_table.csv'
final_candidates_path = OUTPUT_DIR / 'final_policy_candidates_table.csv'
final_decision_path = OUTPUT_DIR / 'final_policy_decision_table.csv'
final_model_path = OUTPUT_DIR / 'final_model_summary_table.csv'
final_rationale_path = OUTPUT_DIR / 'final_policy_rationale_table.csv'
deployment_checklist_path = OUTPUT_DIR / 'final_deployment_checklist_table.csv'
refined_useful_summary_path = OUTPUT_DIR / 'final_refined_useful_summary.csv'

policy_eval_df.to_csv(policy_eval_path, index=False)
final_candidates_df.to_csv(final_candidates_path, index=False)
final_decision_df.to_csv(final_decision_path, index=False)
final_model_summary_df.to_csv(final_model_path, index=False)
final_rationale_df.to_csv(final_rationale_path, index=False)
deployment_checklist_df.to_csv(deployment_checklist_path, index=False)
refined_useful_summary_df.to_csv(refined_useful_summary_path, index=False)

print('Guardado:', policy_eval_path)
print('Guardado:', final_candidates_path)
print('Guardado:', final_decision_path)
print('Guardado:', final_model_path)
print('Guardado:', final_rationale_path)
print('Guardado:', deployment_checklist_path)
print('Guardado:', refined_useful_summary_path)